# Healthcare Vector Store — Semantic Search (TODO 3.2)

Builds a ChromaDB-backed vector store populated with healthcare documents and runs semantic search queries with tracing.

In [ ]:
# ============================================================================
# SETUP
# ============================================================================
!pip install -q chromadb sentence-transformers

In [ ]:
import time
import chromadb
from chromadb.utils import embedding_functions


# ============================================================================
# VectorStore wrapper
# ============================================================================

class HealthcareVectorStore:
    """
    Thin wrapper around ChromaDB that adds per-query tracing/timing.
    """

    def __init__(self):
        self.client = chromadb.Client()   # in-memory client
        self._ef = embedding_functions.DefaultEmbeddingFunction()  # uses all-MiniLM-L6-v2
        self._collections = {}            # name -> chromadb Collection
        self._trace_log = []              # list of trace dicts

    # ------------------------------------------------------------------
    # Collection management
    # ------------------------------------------------------------------

    def get_or_create_collection(self, name: str):
        if name not in self._collections:
            col = self.client.get_or_create_collection(
                name=name,
                embedding_function=self._ef,
                metadata={"hnsw:space": "cosine"}
            )
            self._collections[name] = col
        return self._collections[name]

    def add_documents(self, collection_name: str, documents: list[dict]):
        """
        documents: list of {"id": str, "text": str, "metadata": dict}
        """
        col = self.get_or_create_collection(collection_name)
        col.add(
            ids=[d["id"] for d in documents],
            documents=[d["text"] for d in documents],
            metadatas=[d["metadata"] for d in documents],
        )

    # ------------------------------------------------------------------
    # Search with tracing
    # ------------------------------------------------------------------

    def search_with_tracing(self, collection_name: str, query: str, n_results: int = 2) -> dict:
        """
        Runs a vector search and records a trace entry (query, latency, hit count).
        Returns the raw ChromaDB result dict.
        """
        col = self.get_or_create_collection(collection_name)

        t0 = time.perf_counter()
        results = col.query(
            query_texts=[query],
            n_results=n_results,
            include=["documents", "metadatas", "distances"]
        )
        latency_ms = (time.perf_counter() - t0) * 1000

        hit_count = len(results["documents"][0]) if results["documents"] else 0
        self._trace_log.append({
            "collection": collection_name,
            "query": query,
            "latency_ms": round(latency_ms, 2),
            "hits": hit_count,
        })

        return results

    def get_trace_log(self):
        return self._trace_log


print("HealthcareVectorStore class defined.")

In [ ]:
# ============================================================================
# TODO 3.1 — Populate the healthcare_all collection
# ============================================================================

vector_store = HealthcareVectorStore()

healthcare_documents = [
    # --- Lab / haematology ---
    {
        "id": "hb_001",
        "text": (
            "Normal hemoglobin levels: adult men 13.5–17.5 g/dL, adult women 12.0–15.5 g/dL. "
            "Low hemoglobin (anaemia) may present with fatigue, pallor, and shortness of breath."
        ),
        "metadata": {"filename": "lab_reference_ranges.txt", "category": "haematology"},
    },
    {
        "id": "hb_002",
        "text": (
            "Haemoglobin electrophoresis is used to detect abnormal haemoglobin variants such as HbS "
            "(sickle cell) and HbC. Complete blood count (CBC) includes RBC, WBC, platelets and "
            "haemoglobin measurement."
        ),
        "metadata": {"filename": "lab_reference_ranges.txt", "category": "haematology"},
    },
    # --- Allergy / immunology ---
    {
        "id": "allergy_001",
        "text": (
            "Peanut allergy symptoms include hives, swelling of lips and throat, difficulty breathing, "
            "abdominal pain, and anaphylaxis. Epinephrine (EpiPen) is first-line emergency treatment."
        ),
        "metadata": {"filename": "allergy_clinical_guide.txt", "category": "allergy_immunology"},
    },
    {
        "id": "allergy_002",
        "text": (
            "Ibuprofen (NSAID) allergy or intolerance may cause urticaria, angioedema, bronchospasm, "
            "or anaphylaxis. Patients with aspirin-exacerbated respiratory disease (AERD) often react "
            "to all NSAIDs. Acetaminophen is a common alternative."
        ),
        "metadata": {"filename": "medication_allergy_guide.txt", "category": "allergy_immunology"},
    },
    {
        "id": "allergy_003",
        "text": (
            "Drug allergy testing includes skin-prick tests, intradermal tests, and graded drug "
            "challenges. Penicillin allergy is commonly over-reported; formal evaluation can "
            "de-label up to 90% of patients."
        ),
        "metadata": {"filename": "medication_allergy_guide.txt", "category": "allergy_immunology"},
    },
    # --- Radiology ---
    {
        "id": "xray_001",
        "text": (
            "Chest X-ray findings in pneumonia: lobar or segmental consolidation, air bronchograms, "
            "pleural effusion, and increased opacity. Community-acquired pneumonia most commonly "
            "affects the lower lobes."
        ),
        "metadata": {"filename": "radiology_chest_xray.txt", "category": "radiology"},
    },
    {
        "id": "xray_002",
        "text": (
            "Interstitial lung patterns on chest X-ray include reticular, nodular, and ground-glass "
            "opacities. Bilateral ground-glass in ICU patients may indicate ARDS or COVID-19 "
            "pneumonitis."
        ),
        "metadata": {"filename": "radiology_chest_xray.txt", "category": "radiology"},
    },
    # --- Hepatology / LFTs ---
    {
        "id": "lft_001",
        "text": (
            "Liver function tests (LFTs) include ALT, AST, ALP, GGT, bilirubin, and albumin. "
            "Elevated ALT and AST suggest hepatocellular damage; elevated ALP and GGT suggest "
            "cholestatic disease."
        ),
        "metadata": {"filename": "lft_interpretation_guide.txt", "category": "hepatology"},
    },
    {
        "id": "lft_002",
        "text": (
            "AST:ALT ratio > 2:1 with elevated GGT is characteristic of alcoholic liver disease. "
            "Low albumin and elevated PT/INR indicate impaired hepatic synthetic function and "
            "advanced liver disease."
        ),
        "metadata": {"filename": "lft_interpretation_guide.txt", "category": "hepatology"},
    },
    # --- Cardiology ---
    {
        "id": "cardio_001",
        "text": (
            "Troponin I and troponin T are cardiac biomarkers that rise within 3–6 hours of "
            "myocardial injury and peak at 12–24 hours. They are essential for diagnosing "
            "acute myocardial infarction (AMI)."
        ),
        "metadata": {"filename": "cardiology_biomarkers.txt", "category": "cardiology"},
    },
    # --- Endocrinology ---
    {
        "id": "endo_001",
        "text": (
            "HbA1c (glycated haemoglobin) reflects average blood glucose over 3 months. "
            "Diagnostic threshold for diabetes is ≥ 6.5% (48 mmol/mol). Target for most "
            "diabetic patients is < 7.0% (53 mmol/mol)."
        ),
        "metadata": {"filename": "diabetes_management.txt", "category": "endocrinology"},
    },
]

vector_store.add_documents("healthcare_all", healthcare_documents)
print(f"Loaded {len(healthcare_documents)} documents into 'healthcare_all' collection.")

In [ ]:
# ============================================================================
# TODO 3.2: TEST SEMANTIC SEARCH
# ============================================================================

test_queries = [
    "What are the normal hemoglobin levels?",
    "Patient with peanut allergy symptoms",
    "Chest X-ray findings for pneumonia",
    "Liver function test interpretation",
    "Medication allergy to ibuprofen"
]

print("SEMANTIC SEARCH TEST")
print("="*60)

for i, query in enumerate(test_queries, 1):
    print(f"\n[Query {i}] '{query}'")
    print("-" * 50)

    results = vector_store.search_with_tracing("healthcare_all", query, n_results=2)

    if results['documents'] and results['documents'][0]:
        for j, (doc, meta, dist) in enumerate(zip(
            results['documents'][0],
            results['metadatas'][0],
            results['distances'][0]
        )):
            similarity = 1 - dist if dist else 0
            print(f"  Result {j+1} (Similarity: {similarity:.3f})")
            print(f"  Source: {meta.get('filename', 'N/A')} ({meta.get('category', 'N/A')})")
            print(f"  Preview: {doc[:100]}..." if doc else "  No content")
    else:
        print("  No results (implement search_with_tracing first)")

In [ ]:
# ============================================================================
# Trace summary
# ============================================================================

print("\nTRACE LOG SUMMARY")
print("="*60)
for entry in vector_store.get_trace_log():
    print(f"  [{entry['collection']}] '{entry['query'][:50]}...'  "
          f"latency={entry['latency_ms']}ms  hits={entry['hits']}")